# Macro Risk Cockpit — Methodology

An end-to-end walkthrough of the three modules that drive the dashboard. The notebook is intentionally short on commentary inside cells and longer in markdown — code should read clearly without inline narration.

## 1. Motivation

Single-indicator recession models are easy to read but easy to mislead. The 10-year/3-month yield curve has an outstanding empirical track record but, in 2022–24, gave a much louder signal than realised growth and labor data justified. Survey-based and Street estimates often differ by 20+ percentage points without a transparent decomposition. We respond with three independent lenses that disagree readably:

- A **5-submodel probit ensemble** over yield-curve, labor, credit, housing, and sentiment data.
- A **labor-market composite** (LAME) that summarizes 10 series into a single z-score.
- A **yield-curve module** that exposes the spread term-structure and inversion statistics directly.

An equal-weighted average of the submodels is the headline probability; the dashboard surfaces the components so a reader can see *why* the model is where it is.

## 2. Data

All series are pulled from FRED and stored in `src/data/series_registry.py`. The registry is the single source of truth: each entry carries the FRED ID, native frequency, the transformation applied before modelling, and (for labor indicators) the sign convention so positive numbers always mean *more expansion*.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.data.series_registry import SERIES_REGISTRY

registry_df = pd.DataFrame(SERIES_REGISTRY).T
registry_df.head(15)

## 3. Submodel walkthrough — yield curve

Each submodel applies the same fit recipe:

1. Build features from the FRED panel using the registry transforms.
2. Discard features whose fitted coefficient sign disagrees with the economic prior (e.g. the 10Y−3M spread must enter with a negative coefficient).
3. BIC-stepwise: drop the feature with the smallest |t| until BIC stops improving.
4. Re-check the sign constraints after every drop.

Below we fit the yield-curve submodel against the NBER 12-month forward indicator. The fit is intentionally light — the value of the ensemble is breadth, not estimator sophistication.

In [ ]:
from src.data.fred_client import fetch_panel
from src.data.series_registry import fred_ids
from src.data.nber import load_nber_recessions, recession_in_next_12m
from src.models.recession_ensemble import RecessionEnsemble

panel = fetch_panel(fred_ids(), start='1959-01-01')
nber = load_nber_recessions()
fwd = recession_in_next_12m(nber)

model = RecessionEnsemble()
model.fit(panel, fwd)

current = model.predict_current()
current

In [ ]:
history = model.predict_history()
history.tail()

## 4. Calibration

Brier score and AUC are the standard probabilistic-classifier diagnostics. We compute both on the full in-sample history of the ensemble. The reliability diagram bins predictions into deciles and plots the empirical frequency against the model's predicted probability.

In [ ]:
stats = model.calibration_stats()
print('Brier:', round(stats['brier'], 4))
print('AUC:  ', round(stats['auc'], 3))
stats['reliability_curve']

**Baseline comparison.** A naive baseline that predicts the unconditional 12-month-forward NBER frequency at every date achieves the Brier score below. Our ensemble should comfortably beat it.

In [ ]:
import numpy as np

base_rate = float(fwd.mean())
baseline_brier = float(((fwd.astype(float) - base_rate) ** 2).mean())
print('Baseline (unconditional) Brier:', round(baseline_brier, 4))
print('Ensemble Brier:                 ', round(stats['brier'], 4))

## 5. Comparison to published models

The dashboard's *vs. Street* tab compares our reading against Cleveland Fed, NY Fed, Bloomberg, and Goldman estimates. Those values live in `data/street_estimates.csv` and are maintained manually — the cell below loads them and aligns to the most recent month.

In [ ]:
street = pd.read_csv('../data/street_estimates.csv', parse_dates=['date']).sort_values('date')
street.tail()

When this notebook is re-run, the values will refresh automatically from FRED. Edit `data/street_estimates.csv` to update the Street comparison.